# MERFISH Schrödinger Bridge: Cell-Type Spatial Evaluation

End-to-end pipeline on the squidpy MERFISH dataset (Moffitt et al. 2018, mouse hypothalamus):

1. Load data, split into train / held-out sections
2. Compute entropic OT couplings between consecutive slices
3. Train a Schrödinger bridge via IPF
4. Reconstruct held-out z-planes via bidirectional SDE
5. **Visualise** ground-truth vs. reconstructed cell-type spatial maps
6. Quantitative evaluation (Sinkhorn, NN-MSE, Pearson)

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)
torch.manual_seed(42)

print(f'PyTorch {torch.__version__}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Load MERFISH data and split into train / held-out

In [ ]:
from st3d.load_merfish import load_merfish

split = load_merfish(holdout_every=3)

### Detect cell-type column

The MERFISH dataset typically annotates cells with a `Cell_class` column.
We auto-detect the best categorical annotation column to use for colouring.

In [ ]:
# Auto-detect the cell-type annotation column
sample_section = split.heldout_sections[0]
obs_cols = list(sample_section.obs.columns)
print(f'Available obs columns: {obs_cols}')

CELLTYPE_COL = None
# Priority order of candidate column names
for candidate in ['Cell_class', 'cell_type', 'celltype', 'cluster',
                   'cell_class', 'CellType', 'leiden', 'louvain']:
    if candidate in obs_cols:
        CELLTYPE_COL = candidate
        break

# Fallback: pick the first categorical/object column with a reasonable
# number of unique values (2-50)
if CELLTYPE_COL is None:
    for col in obs_cols:
        dtype = sample_section.obs[col].dtype
        if dtype == object or hasattr(dtype, 'categories'):
            n_unique = sample_section.obs[col].nunique()
            if 2 <= n_unique <= 50:
                CELLTYPE_COL = col
                break

if CELLTYPE_COL is None:
    print('No cell-type column found -- will cluster with Leiden.')
    import scanpy as sc
    import squidpy as sq
    adata_full = sq.datasets.merfish()
    sc.pp.normalize_total(adata_full, target_sum=1e4)
    sc.pp.log1p(adata_full)
    sc.pp.pca(adata_full, n_comps=30)
    sc.pp.neighbors(adata_full, n_pcs=15)
    sc.tl.leiden(adata_full, resolution=0.5, key_added='leiden')
    CELLTYPE_COL = 'leiden'
    # Propagate labels back into train/heldout sections by index
    for sec in split.train_sections + split.heldout_sections:
        common = sec.obs.index.intersection(adata_full.obs.index)
        sec.obs[CELLTYPE_COL] = adata_full.obs.loc[common, CELLTYPE_COL].values
else:
    print(f'Using cell-type column: "{CELLTYPE_COL}"')

# Show distribution in first held-out section
print(f'\nCell-type distribution (first held-out section):')
print(split.heldout_sections[0].obs[CELLTYPE_COL].value_counts())

In [ ]:
# Build a consistent colour palette across all sections
all_types = set()
for sec in split.train_sections + split.heldout_sections:
    all_types.update(sec.obs[CELLTYPE_COL].unique())
all_types = sorted(all_types)

cmap = plt.cm.get_cmap('tab20', len(all_types))
PALETTE = {ct: cmap(i) for i, ct in enumerate(all_types)}

print(f'{len(all_types)} cell types: {all_types}')

## 2. Compute entropic OT couplings

In [ ]:
from st3d.load_merfish import compute_transport_plans

plans = compute_transport_plans(split.train_tensors, blur=0.05, n_iters=100)

## 3. Train the Schrödinger bridge (IPF)

In [ ]:
from st3d.bridge import IPFTrainer, IPFConfig

cfg = IPFConfig(
    hidden_dim=256,
    n_blocks=3,
    time_embed_dim=32,
    sigma=0.1,
    batch_size=512,
    n_ipf_iters=5,
    steps_per_half=200,
    sde_steps=30,
    ot_blur=0.05,
    ot_iters=100,
    lr=1e-3,
)

trainer = IPFTrainer(split.train_tensors, plans, config=cfg, device=device)
print(f'Model parameters: {sum(p.numel() for p in trainer.model.parameters()):,}')
history = trainer.fit(verbose=True)

### Training loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['fwd_loss'], linewidth=0.5)
axes[0].set_title('Forward bridge loss')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('MSE')

axes[1].plot(history['bwd_loss'], linewidth=0.5, color='tab:orange')
axes[1].set_title('Backward bridge loss')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('MSE')

for ax in axes:
    ax.set_yscale('log')

fig.tight_layout()
plt.show()

## 4. Predict held-out slices and transfer cell-type labels

For each held-out section we:
1. Find its two flanking training sections
2. Simulate the forward and backward SDE to the held-out depth
3. Blend bidirectionally
4. Transfer cell-type labels from the **ground-truth held-out section** to
   predicted cells via nearest-neighbour spatial matching (for visualisation only)

In [ ]:
from st3d.evaluate import predict_at_z
from st3d.data import denormalize_coords

model = trainer.model
model.eval()

# For each held-out section, predict and collect results
predictions = []  # list of (pred_tensor, true_tensor, bregma, h_idx)

for h_idx, (h_tensor, h_z) in enumerate(zip(split.heldout_tensors, split.heldout_z)):
    # Find flanking training sections
    left_idx, right_idx = None, None
    for k, tz in enumerate(split.train_z):
        if tz <= h_z:
            left_idx = k
        if tz >= h_z and right_idx is None:
            right_idx = k

    if left_idx is None or right_idx is None or left_idx == right_idx:
        print(f'  [SKIP] Held-out {h_idx}: z={h_z:.3f} outside training range')
        continue

    z_left = split.train_z[left_idx]
    z_right = split.train_z[right_idx]
    alpha = (h_z - z_left) / (z_right - z_left)

    pred = predict_at_z(
        model,
        split.train_tensors[left_idx],
        split.train_tensors[right_idx],
        alpha=alpha,
        sigma=cfg.sigma,
        sde_steps=50,
        device=torch.device(device),
    )

    bregma = split.heldout_bregma[h_idx]
    predictions.append((pred.cpu(), h_tensor, bregma, h_idx))
    print(f'  Held-out {h_idx}: Bregma {bregma:+.2f}  z={h_z:.3f}  '
          f'alpha={alpha:.3f}  pred={pred.shape[0]} cells  true={h_tensor.shape[0]} cells')

print(f'\n{len(predictions)} held-out slices predicted.')

## 5. Cell-type spatial maps: Ground truth vs. Reconstructed

Each row is one held-out slice. **Left** = ground truth, **Right** = model
prediction. Cell-type labels on predicted cells are transferred by
spatially matching each predicted cell to its nearest ground-truth neighbour.

In [ ]:
def plot_celltype_spatial(ax, xy, labels, palette, title, point_size=3):
    """Scatter plot coloured by cell type."""
    for ct in sorted(palette.keys()):
        mask = labels == ct
        if mask.sum() == 0:
            continue
        ax.scatter(
            xy[mask, 0], xy[mask, 1],
            c=[palette[ct]], s=point_size, alpha=0.7,
            label=ct, rasterized=True,
        )
    ax.set_title(title, fontsize=11)
    ax.set_aspect('equal')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')


n_plots = len(predictions)
fig, axes = plt.subplots(n_plots, 2, figsize=(14, 6 * n_plots), squeeze=False)

norm_params = split.norm_params

for row, (pred_tensor, true_tensor, bregma, h_idx) in enumerate(predictions):
    # --- Ground truth ---
    gt_section = split.heldout_sections[h_idx]
    gt_xy = gt_section.obsm['spatial'][:, :2]
    gt_labels = gt_section.obs[CELLTYPE_COL].values

    plot_celltype_spatial(
        axes[row, 0], gt_xy, gt_labels, PALETTE,
        title=f'Ground Truth  (Bregma {bregma:+.2f},  {len(gt_labels)} cells)',
    )

    # --- Predicted ---
    # Denormalise predicted xy coords back to original scale
    pred_xy_norm = pred_tensor[:, :2].numpy()
    pred_xy = denormalize_coords(
        np.column_stack([pred_xy_norm, np.zeros(len(pred_xy_norm))]),
        norm_params.xy_min, norm_params.xy_scale,
    )[:, :2]

    # Transfer cell-type labels from ground truth via spatial NN
    dists = torch.cdist(
        torch.from_numpy(pred_xy).float(),
        torch.from_numpy(gt_xy.astype(np.float32)),
    )
    nn_idx = dists.argmin(dim=1).numpy()
    pred_labels = gt_labels[nn_idx]

    plot_celltype_spatial(
        axes[row, 1], pred_xy, pred_labels, PALETTE,
        title=f'Reconstructed  (Bregma {bregma:+.2f},  {len(pred_labels)} cells)',
    )

# Shared legend
handles = [mpatches.Patch(color=PALETTE[ct], label=ct) for ct in sorted(PALETTE.keys())]
fig.legend(
    handles=handles, loc='center left', bbox_to_anchor=(1.0, 0.5),
    fontsize=8, title=CELLTYPE_COL, title_fontsize=9,
)

fig.suptitle('Cell-type spatial maps: Ground Truth vs. Reconstructed', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

### Per-cell-type density comparison

For each held-out slice, compare the spatial density of each cell type between
ground truth and reconstruction using per-type point counts in a grid.

In [ ]:
# Cell-type composition bar chart: ground truth vs predicted
for pred_tensor, true_tensor, bregma, h_idx in predictions:
    gt_section = split.heldout_sections[h_idx]
    gt_labels = gt_section.obs[CELLTYPE_COL].values
    gt_xy = gt_section.obsm['spatial'][:, :2]

    # Predicted labels via NN transfer
    pred_xy_norm = pred_tensor[:, :2].numpy()
    pred_xy = denormalize_coords(
        np.column_stack([pred_xy_norm, np.zeros(len(pred_xy_norm))]),
        norm_params.xy_min, norm_params.xy_scale,
    )[:, :2]
    dists = torch.cdist(
        torch.from_numpy(pred_xy).float(),
        torch.from_numpy(gt_xy.astype(np.float32)),
    )
    nn_idx = dists.argmin(dim=1).numpy()
    pred_labels = gt_labels[nn_idx]

    # Count proportions
    types = sorted(PALETTE.keys())
    gt_frac = np.array([np.mean(gt_labels == ct) for ct in types])
    pred_frac = np.array([np.mean(pred_labels == ct) for ct in types])

    # Only show types that have > 1% in either
    mask = (gt_frac > 0.01) | (pred_frac > 0.01)
    types_show = [t for t, m in zip(types, mask) if m]
    gt_show = gt_frac[mask]
    pred_show = pred_frac[mask]

    x = np.arange(len(types_show))
    width = 0.35

    fig, ax = plt.subplots(figsize=(max(8, len(types_show) * 0.8), 4))
    ax.bar(x - width/2, gt_show, width, label='Ground Truth',
           color=[PALETTE[t] for t in types_show], edgecolor='black', linewidth=0.5)
    ax.bar(x + width/2, pred_show, width, label='Reconstructed',
           color=[PALETTE[t] for t in types_show], edgecolor='black', linewidth=0.5,
           alpha=0.5, hatch='//')
    ax.set_xticks(x)
    ax.set_xticklabels(types_show, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Fraction of cells')
    ax.set_title(f'Cell-type composition  (Bregma {bregma:+.2f})')
    ax.legend()
    fig.tight_layout()
    plt.show()

## 6. Quantitative evaluation

Sinkhorn distance, NN-MSE, and Pearson correlation across all held-out slices.

In [ ]:
from st3d.evaluate import evaluate_heldout, print_summary

results = evaluate_heldout(
    trainer.model,
    train_tensors=split.train_tensors,
    train_z=split.train_z,
    heldout_tensors=split.heldout_tensors,
    heldout_z=split.heldout_z,
    sigma=cfg.sigma,
    sde_steps=50,
    sinkhorn_blur=0.05,
    sinkhorn_iters=100,
    device=device,
    heldout_bregma=split.heldout_bregma,
)

agg = print_summary(results)

### Per-slice Pearson correlation by PC

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for m in results:
    label = f'Bregma {m.bregma:+.2f}' if m.bregma is not None else f'z={m.z:.3f}'
    ax.plot(m.pearson_per_pc, marker='o', markersize=3, linewidth=1, label=label)

ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Pearson r')
ax.set_title('Per-PC Pearson correlation (predicted vs. ground truth)')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### Metrics bar chart

In [ ]:
if len(results) > 1:
    labels = [f'{m.bregma:+.2f}' if m.bregma else f'{m.z:.3f}' for m in results]
    x = np.arange(len(labels))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Sinkhorn (spatial)
    vals = [m.sinkhorn_spatial for m in results]
    axes[0].bar(x, vals, color='steelblue')
    axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
    axes[0].set_title('Sinkhorn distance (xy)')
    axes[0].set_xlabel('Bregma')

    # NN-MSE (expression)
    vals = [m.nn_mse_expr for m in results]
    axes[1].bar(x, vals, color='coral')
    axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
    axes[1].set_title('NN-MSE (expression PCs)')
    axes[1].set_xlabel('Bregma')

    # Pearson
    vals = [m.pearson_mean for m in results]
    axes[2].bar(x, vals, color='mediumseagreen')
    axes[2].set_xticks(x); axes[2].set_xticklabels(labels)
    axes[2].set_title('Mean Pearson r')
    axes[2].set_xlabel('Bregma')
    axes[2].set_ylim([-0.1, 1.0])

    fig.suptitle('Evaluation metrics per held-out slice', fontsize=13)
    fig.tight_layout()
    plt.show()
else:
    print('Only one held-out slice -- skipping bar chart.')

---

**Summary of visualisations:**

| Plot | Purpose |
|---|---|
| Side-by-side spatial maps | Do predicted cells land where the correct cell types are? |
| Cell-type composition bars | Does the model preserve the global proportions of each type? |
| Per-PC Pearson | Which expression programs are best captured by the bridge? |
| Metrics bar chart | Which held-out slices are easiest / hardest to reconstruct? |